# Pipeline Leak Detection - Quick Start

This notebook demonstrates how to use the Conformer-based leak detection model.

In [ ]:
import sys
sys.path.append('../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
import yaml

from leak_detection.models import ConformerLeakDetector
from leak_detection.data import LeakAudioDataset

## 1. Load Configuration

In [ ]:
with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Model Configuration:")
print(f"  d_model: {config['model']['d_model']}")
print(f"  num_layers: {config['model']['num_layers']}")
print(f"  num_heads: {config['model']['num_heads']}")

## 2. Create Model

In [ ]:
model = ConformerLeakDetector(config['model'])

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 3. Test Forward Pass

In [ ]:
# Create dummy input
batch_size = 4
time_steps = 200
n_mels = 128

dummy_input = torch.randn(batch_size, time_steps, n_mels)

# Forward pass
detection_logits, distance_pred, shape_logits = model(dummy_input)

print(f"Detection logits shape: {detection_logits.shape}")
print(f"Distance prediction shape: {distance_pred.shape}")
print(f"Shape logits shape: {shape_logits.shape}")

## 4. Test Inference Mode

In [ ]:
predictions = model.predict(dummy_input)

print("Predictions:")
for key, value in predictions.items():
    print(f"  {key}: {value}")

## 5. Visualize Model Architecture

In [ ]:
# Plot model structure
print("\nModel Architecture:")
print(model)

## 6. Training Example (Conceptual)

```bash
# Train the model
python ../src/train.py --config ../configs/config.yaml

# Evaluate
python ../src/evaluate.py --checkpoint outputs/checkpoint_best.pth

# Inference
python ../src/inference.py --audio path/to/audio.wav --checkpoint outputs/checkpoint_best.pth
```